[Guide Index](README.md)


# 05c SRL (SPARQL Rules) as RDF

A parsed SRL rule set — `srl.RuleSet`, the same dataclass tree `srl.parse_ruleset()` returns (see the [SRL (SPARQL Rules) guide](03b-srl-sparql-rules.ipynb) for the rules language itself) — can also be represented as an RDF graph, under the `srl:` namespace, the same "syntax-tree-as-RDF" pattern used elsewhere in StarLayer (e.g. [SPARQL queries as RDF](05b-sparql-query-as-rdf.ipynb), [Manchester Syntax](05a-manchester-syntax.ipynb)). This lets a rule set be stored, inspected, or edited as ordinary RDF, and validated structurally with SHACL.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections reuse variables from earlier ones.


## 1. A parsed rule set

The same base graph and rule set as the [SRL (SPARQL Rules) guide](03b-srl-sparql-rules.ipynb) - see that guide for what these rules mean; this one is only about representing the already-parsed result as RDF.


In [1]:
from starlayer.graph import StarLayerGraph, Namespace
from starlayer.sparql import srl

EX = Namespace("http://example.org/")

base = StarLayerGraph()
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")
base.bind("ex", EX)

rules_text = """
PREFIX ex: <http://example.org/>

RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = srl.parse_ruleset(rules_text)
print(f"{len(ruleset.rules)} rules parsed")


4 rules parsed


## 2. Encode the rule set as RDF

`ruleset_to_tree()` encodes a `RuleSet` as RDF under the `srl:` namespace - the same tree-RDF pattern `sqe:` uses for SPARQL queries (see the [SPARQL queries as RDF guide](05b-sparql-query-as-rdf.ipynb)). It returns `(graph, root)` - `root` is the `srl:RuleSet` node's own identity, needed later to decode a specific rule set back out of the graph.

Each rule, and each `NOT`, carries the spec's `evaldata` flag as `srl:ruleEvaldata` or `srl:negationEvaldata`: `true` means the patterns match the evaluation graph (a plain `WHERE` or `NOT`), `false` means they match only the base graph (`WHERE DATA` or `NOT DATA`; see the [SRL (SPARQL Rules) guide](03b-srl-sparql-rules.ipynb)).

In [2]:
from starlayer.sparql.vocab import SPEXPR

srl_graph, root = srl.ruleset_to_tree(ruleset)

srl_graph.bind("srl", srl.SRL)
srl_graph.bind("spexpr", SPEXPR)
srl_graph.bind("ex", EX)

print("ruleset as graph:")
print(srl_graph.serialize())

ruleset as graph:
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix srl: <https://github.com/hidden-graph/starlayer/ns/sparql/srl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

_:Na9cc98c5119c4622b15f0314fb2de3bb a srl:RuleSet ;
    srl:data () ;
    srl:imports () ;
    srl:rules ( [
        a srl:Rule ;
        srl:body ( [
            a srl:TriplePattern ;
            srl:object "v"^^spexpr:Variable ;
            srl:predicate ex:hasVulnerability ;
            srl:subject "x"^^spexpr:Variable
        ] ) ;
        srl:head ( [
            a srl:TriplePattern ;
            srl:object "v"^^spexpr:Variable ;
            srl:predicate ex:exposedTo ;
            srl:subject "x"^^spexpr:Variable
        ] ) ;
        srl:ruleEvaldata true
    ] [
        a srl:Rule ;
        srl:body ( [
            a srl:TriplePattern ;
            srl:object "y"^^

## 3. Edit the rule set by editing the RDF

The rule set graph can be edited as an RDF graph. Below, rule 2's own body - `?x ex:dependsOn ?y . ?y ex:exposedTo ?v` - is edited so the dependency edge it walks is `ex:reliesOn` instead of `ex:dependsOn`, directly as RDF graph surgery, no text-level find/replace anywhere.

In [3]:
# a triple pattern's own predicate is located and replaced
for s in list(srl_graph.subjects(srl.SRL.predicate, EX.dependsOn)):
    srl_graph.set((s, srl.SRL.predicate, EX.reliesOn))

print("edited: rule 2's body now walks ex:reliesOn, not ex:dependsOn")

edited: rule 2's body now walks ex:reliesOn, not ex:dependsOn


## 4. Decode back into a real `RuleSet`, and run it

`tree_to_ruleset()` turns the edited RDF back into a real `RuleSet`. The base data's dependency chain is `frontend -> app -> db`, with the only vulnerability on `db` itself - running both rule sets against the *same* base graph proves the edit is semantic: breaking the `ex:dependsOn` propagation edge stops `app`/`frontend` from inheriting `db`'s exposure, so only `db` stays `criticallyExposed`.

In [4]:
edited_ruleset = srl.tree_to_ruleset(srl_graph, root)

original_inferred = ruleset.infer(base)
edited_inferred = edited_ruleset.infer(base)

print("original criticallyExposed:", sorted(base.qname(s) for s in original_inferred.subjects(EX.status, EX.criticallyExposed)))
print("edited criticallyExposed:  ", sorted(base.qname(s) for s in edited_inferred.subjects(EX.status, EX.criticallyExposed)))

original criticallyExposed: ['ex:app', 'ex:db', 'ex:frontend']
edited criticallyExposed:   ['ex:db']


## 5. Export the modified rule set back to text

`ruleset_to_text()` renders the edited `RuleSet` back to SRL text. Unlike `sqe_tree_to_text()`'s own `PREFIX` preservation, the parsed SRL syntax tree only ever holds fully-resolved absolute IRIs - no original `PREFIX`/`BASE` declaration survives to render back. Passing a `namespace_manager` (here, the base graph's own `ex:` binding) at least abbreviates the output; without one, every IRI renders in full.

In [5]:
print(srl.ruleset_to_text(edited_ruleset, namespace_manager=base.namespace_manager))

RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:reliesOn ?y . ?y ex:exposedTo ?v . }
RULE { ?x ex:status ex:criticallyExposed . } WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= "9.0"^^<http://www.w3.org/2001/XMLSchema#decimal>) }
RULE { ?x ex:status ex:safeToDeploy . } WHERE { ?x rdf:type ex:Component . NOT { ?x ex:status ex:criticallyExposed . } }


## 6. Validate — automatically on parse, and again after editing outside the parse

`parse_ruleset()` already validates every rule set it parses, first against §4.2's own well-formedness conditions (unbound head/filter/assignment variables, a `SET(...)` reusing an already-bound variable, an aggregate used inside `FILTER` or `SET`) and raises `SRLParseError` if any is violated - see 6.1 below. It then checks the rule set against the `srl:` SHACL shapes, the same structural check `srl_validate(data_graph)` runs over an `srl:` tree-RDF graph, and raises `SRLParseError` if the rule set does not conform (pass `validate=False` to skip that second step; it takes about a tenth of a second). Both checks only cover the moment of parsing, though. `srl_validate()` is still needed whenever a tree is edited by hand after parsing, since those edits (like section 3's above) happen outside of `parse_ruleset()` and aren't automatically re-checked.

In [6]:
# validate the edited syntax tree from section 3 above
conforms, _report_graph, report_text = srl.srl_validate(srl_graph)
print("conforms:", conforms)

conforms: True


Editing the tree can also make it invalid. Removing a `srl:TriplePattern`'s own required `srl:predicate` is one example - `srl_validate()` catches it:

In [7]:
from rdflib import RDF

# break the tree: remove a triple pattern's own required predicate
pattern_node = next(srl_graph.subjects(RDF.type, srl.SRL.TriplePattern))
srl_graph.remove((pattern_node, srl.SRL.predicate, None))

conforms, broken_report_graph, broken_report_text = srl.srl_validate(srl_graph)
print("conforms:", conforms)
print(broken_report_text)

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:maxCount Literal("1", datatype=xsd:integer) ; sh:message Literal("srl:TriplePattern must have exactly one srl:predicate, an IRI/Variable") ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:node srl:PredicateOrVariableShape ; sh:path srl:predicate ]
	Focus Node: [ rdf:type rdfs:Resource, srl:BodyElement, srl:NegationBodyElement, srl:TriplePattern ; srl:object Literal("v", datatype=spexpr:Variable) ; srl:subject Literal("x", datatype=spexpr:Variable) ]
	Result Path: srl:predicate
	Message: srl:TriplePattern must have exactly one srl:predicate, an IRI/Variable

Provenance:
	Used Shapes Graph: starontology:srl_shapes.ttl


### 6.1 A rule set that's well-formed SPARQL-wise, but fails SRL's own §4.2 check

`RULE { ?x ex:status ?z } WHERE { ?x a ex:Component . }` parses as a perfectly ordinary rule - but `?z` in the head is never bound anywhere in the body. §4.2's well-formedness condition requires every head variable to appear in the body's own bound-variable set; `?z` doesn't, so this signals a typo or an incomplete rule, the same "output with no matching input" shape as `sqe:`'s own unbound-projected-variable check in the [SPARQL queries as RDF guide](05b-sparql-query-as-rdf.ipynb).

Because `parse_ruleset()` validates automatically, parsing this text raises `SRLParseError` directly.

In [8]:
bad_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:status ?z } WHERE { ?x a ex:Component . }
"""

try:
    srl.parse_ruleset(bad_rules)
except srl.SRLParseError as exc:
    print("raised:", exc)

raised: SRL: rule set is not well-formed per §4.2: unbound_head_variable (z in rule head)
